### Setup

In [1]:
%load_ext autoreload
%autoreload 2

import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from app.config import get_settings

settings = get_settings()
os.environ["HF_TOKEN"] = settings.hf_token

print({k: ("set" if v else "MISSING") for k, v in settings.model_dump().items()})

{'app_env': 'set', 'database_url': 'set', 'groq_api_key': 'set', 'groq_model': 'set', 'hf_token': 'set', 'embedding_model': 'set', 'fernet_key': 'set', 'llm_cache_dir': 'set', 'mem0_api_key': 'set', 'demo_database_url': 'set'}


In [2]:
from sqlalchemy import create_engine, text

engine = create_engine(settings.database_url, pool_pre_ping=True)


def sql(query: str, **params) -> pd.DataFrame:
    """Run a query on the app database and return a DataFrame."""
    with engine.connect() as conn:
        res = conn.execute(text(query), params)
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


display(sql("select version()"))
display(sql("select extname, extversion from pg_extension where extname = 'vector'"))

,version
0,"PostgreSQL 17.11 on aarch64-unknown-linux-gnu,..."


,extname,extversion
0,vector,0.8.2


In [3]:
from groq import APIConnectionError, Groq, RateLimitError

client = Groq(api_key=settings.groq_api_key)

In [4]:
CACHE_DIR = Path(settings.llm_cache_dir)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def llm(messages, *, model=None, temperature=0.0, json_mode=False, use_cache=True, **kwargs) -> str:
    """Single entry point for all LLM calls. Cached on disk so reruns are free and reproducible."""
    model = model or settings.groq_model
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "json_mode": json_mode,
        **kwargs,
    }
    key = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    path = CACHE_DIR / f"{key}.json"
    if use_cache and path.exists():
        return json.loads(path.read_text(encoding="utf-8"))["text"]

    params = dict(model=model, messages=messages, temperature=temperature, **kwargs)
    if json_mode:
        params["response_format"] = {"type": "json_object"}

    for attempt in range(4):
        try:
            resp = client.chat.completions.create(**params)
            break
        except (RateLimitError, APIConnectionError):
            if attempt == 3:
                raise
            time.sleep(2**attempt)

    text_out = resp.choices[0].message.content
    path.write_text(json.dumps({"text": text_out}), encoding="utf-8")
    return text_out


msgs = [{"role": "user", "content": "Reply with exactly one word: ready"}]
for attempt in (1, 2):
    t0 = time.perf_counter()
    print(attempt, llm(msgs), f"{time.perf_counter() - t0:.2f}s")

1 ready 0.00s
2 ready 0.00s


In [5]:
import httpx

get_settings.cache_clear()
settings = get_settings()

HF_EMBED_URL = (
    "https://router.huggingface.co/hf-inference/models/"
    f"{settings.embedding_model}/pipeline/feature-extraction"
)

EMB_DIR = Path(settings.llm_cache_dir).parent / "emb"
EMB_DIR.mkdir(parents=True, exist_ok=True)


def _emb_path(text: str) -> Path:
    key = hashlib.sha256(f"{settings.embedding_model}\n{text}".encode()).hexdigest()
    return EMB_DIR / f"{key}.json"


def _hf_embed_batch(texts: list[str], max_retries: int = 4) -> list[list[float]]:
    headers = {"Authorization": f"Bearer {settings.hf_token}"}
    with httpx.Client(timeout=30) as http:
        for attempt in range(max_retries):
            resp = http.post(HF_EMBED_URL, headers=headers, json={"inputs": texts})
            if resp.status_code == 200:
                data = resp.json()
                ok = (
                    isinstance(data, list)
                    and data
                    and isinstance(data[0], list)
                    and isinstance(data[0][0], (int, float))
                )
                if not ok:
                    raise ValueError(f"Unexpected embedding shape from HF: {str(data)[:200]}")
                return data
            if resp.status_code in (429, 503):  # rate limited, or model cold-starting
                wait = 2 ** (attempt + 1)
                if resp.status_code == 503:
                    try:
                        wait = min(float(resp.json().get("estimated_time", wait)), 20)
                    except Exception:  # noqa: S110
                        pass
                print(f"HF returned {resp.status_code}, retrying in {wait:.0f}s")
                time.sleep(wait)
                continue
            raise RuntimeError(f"HF embedding error {resp.status_code}: {resp.text[:300]}")
    raise RuntimeError(f"HF embedding API failed after {max_retries} retries")


def embed(texts: list[str], *, use_cache: bool = True, batch_size: int = 32) -> list[list[float]]:
    """Embed texts, caching each one on disk so reruns are free and reproducible."""
    out = [None] * len(texts)
    missing = []
    for i, t in enumerate(texts):
        p = _emb_path(t)
        if use_cache and p.exists():
            out[i] = json.loads(p.read_text(encoding="utf-8"))
        else:
            missing.append(i)
    for s in range(0, len(missing), batch_size):
        idx = missing[s : s + batch_size]
        vecs = _hf_embed_batch([texts[i] for i in idx])
        for i, v in zip(idx, vecs):
            out[i] = v
            _emb_path(texts[i]).write_text(json.dumps(v), encoding="utf-8")
    return out


def embed_query(text: str) -> list[float]:
    return embed([text])[0]


def cos(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


vecs = embed(["customer orders", "product price", "customer purchases"])
print("vectors:", len(vecs), "| dimension:", len(vecs[0]))
print("orders vs purchases:", round(cos(vecs[0], vecs[2]), 3))
print("orders vs price:    ", round(cos(vecs[0], vecs[1]), 3))

vectors: 3 | dimension: 384
orders vs purchases: 0.738
orders vs price:     0.247


### Demo data

In [6]:
tables = ["customers", "categories", "products", "orders", "order_items", "payments", "reviews"]
display(
    pd.DataFrame(
        {
            "table": tables,
            "rows": [int(sql(f"select count(*) as c from demo.{t}")["c"][0]) for t in tables],
        }
    )
)

,table,rows
0,customers,500
1,categories,15
2,products,120
3,orders,3000
4,order_items,6252
5,payments,2958
6,reviews,1184


In [7]:
demo_engine = create_engine(settings.demo_database_url, pool_pre_ping=True)


def demo_sql(query: str, **params) -> pd.DataFrame:
    """Run a query on the demo data through the read-only role."""
    with demo_engine.connect() as conn:
        res = conn.execute(text(query), params)
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


display(
    demo_sql("""
    select current_user,
           current_setting('search_path') as search_path,
           current_setting('default_transaction_read_only') as read_only,
           current_setting('statement_timeout') as statement_timeout
""")
)

,current_user,search_path,read_only,statement_timeout
0,askdata_ro,demo,on,15s


In [8]:
for stmt in [
    "insert into customers (customer_id, full_name, email, signup_date, cust_stat_cd) "
    "values (999999, 'x', 'x@example.com', '2026-01-01', 'A')",
    "update orders set ord_sts = 'C'",
    "drop table customers",
]:
    try:
        with demo_engine.begin() as conn:
            conn.execute(text(stmt))
        print("!! WAS ALLOWED:", stmt[:50])
    except Exception as e:
        print("refused:", str(getattr(e, "orig", e)).splitlines()[0])

refused: cannot execute INSERT in a read-only transaction
refused: cannot execute UPDATE in a read-only transaction
refused: cannot execute DROP TABLE in a read-only transaction


In [9]:
display(
    demo_sql("select ord_sts, count(*) as orders from orders group by ord_sts order by orders desc")
)

display(
    demo_sql("""
    select c.customer_id, c.full_name, round(sum(o.total_amount), 2) as revenue
    from customers c
    join orders o on o.customer_id = c.customer_id
    where o.ord_sts = 'D'
    group by c.customer_id, c.full_name
    order by revenue desc
    limit 5
""")
)

display(
    demo_sql("""
    select date_trunc('month', order_date)::date as month, round(sum(total_amount), 2) as revenue
    from orders
    where ord_sts = 'D'
    group by 1
    order by 1
""")
)

,ord_sts,orders
0,D,2312
1,C,372
2,R,239
3,P,42
4,S,35


,customer_id,full_name,revenue
0,461,Neel Mani,18658077.15
1,389,Neel Sharaf,5785237.00
2,426,Faqid Wason,3326826.55
3,274,Urishilla Saran,2427404.05
4,244,Samar Bali,1975657.90


,month,revenue
0,2025-01-01,3029376.25
1,2025-02-01,1754115.65
2,2025-03-01,2122381.35
3,2025-04-01,1959318.15
4,2025-05-01,1964747.15
5,2025-06-01,1497545.00
6,2025-07-01,2262276.50
7,2025-08-01,3127553.55
8,2025-09-01,3142631.35
9,2025-10-01,3221696.75


### Schema introspection and Describer agent

In [10]:
from sqlalchemy import inspect as sa_inspect

from app.schemas.contracts import (
    ColumnDescription,
    EntityInfo,
    FieldInfo,
    RelationshipInfo,
    SchemaSnapshot,
    TableDescription,
    ValueCount,
)

PII_HINTS = (
    "email",
    "phone",
    "mobile",
    "full_name",
    "first_name",
    "last_name",
    "address",
    "aadhaar",
    "passport",
)


def is_personal(col_name: str) -> bool:
    return any(h in col_name.lower() for h in PII_HINTS)


def introspect_postgres(
    eng, schema: str, connection_id: str = "demo", top_n: int = 8, enum_threshold: int = 25
) -> SchemaSnapshot:
    """Read tables, columns, keys and value profiles into a SchemaSnapshot (no LLM involved)."""
    insp = sa_inspect(eng)
    prep = eng.dialect.identifier_preparer
    entities, relationships = [], []

    for table in sorted(insp.get_table_names(schema=schema)):
        pk = set(insp.get_pk_constraint(table, schema=schema).get("constrained_columns") or [])
        fk_ref = {}
        for fk in insp.get_foreign_keys(table, schema=schema):
            for col, rcol in zip(fk["constrained_columns"], fk["referred_columns"]):
                fk_ref[col] = f"{fk['referred_table']}.{rcol}"
                relationships.append(
                    RelationshipInfo(
                        source=table,
                        target=fk["referred_table"],
                        via=f"{table}.{col} -> {fk['referred_table']}.{rcol}",
                    )
                )

        qt = f"{prep.quote_schema(schema)}.{prep.quote(table)}"
        fields = []
        with eng.connect() as conn:
            row_count = conn.execute(text(f"select count(*) from {qt}")).scalar()
            for col in insp.get_columns(table, schema=schema):
                qc = prep.quote(col["name"])
                personal = is_personal(col["name"])
                distinct = conn.execute(text(f"select count(distinct {qc}) from {qt}")).scalar()
                top = []
                if not personal and distinct <= enum_threshold:
                    # low cardinality: list the values with counts (this is where codes show up)
                    rows = conn.execute(
                        text(
                            f"select {qc}::text, count(*) from {qt} where {qc} is not null "
                            f"group by 1 order by 2 desc limit {top_n}"
                        )
                    ).fetchall()
                    top = [ValueCount(value=v, count=c) for v, c in rows]
                elif not personal:
                    # high cardinality: just a few examples
                    rows = conn.execute(
                        text(f"select distinct {qc}::text from {qt} where {qc} is not null limit 3")
                    ).fetchall()
                    top = [ValueCount(value=r[0]) for r in rows]
                fields.append(
                    FieldInfo(
                        name=col["name"],
                        type=str(col["type"]),
                        nullable=col["nullable"],
                        primary_key=col["name"] in pk,
                        references=fk_ref.get(col["name"]),
                        distinct_count=distinct,
                        top_values=top,
                        sensitive=personal,
                    )
                )
        entities.append(EntityInfo(name=table, row_count=row_count, fields=fields))

    return SchemaSnapshot(
        connection_id=connection_id,
        engine="postgres",
        family="sql",
        namespace=schema,
        entities=entities,
        relationships=relationships,
    )

In [11]:
snap = introspect_postgres(demo_engine, "demo")

print(
    f"{len(snap.entities)} tables, "
    f"{sum(len(e.fields) for e in snap.entities)} fields, "
    f"{len(snap.relationships)} relationships"
)

display(
    pd.DataFrame(
        [{"table": e.name, "rows": e.row_count, "columns": len(e.fields)} for e in snap.entities]
    )
)
display(pd.DataFrame([{"relationship": r.via} for r in snap.relationships]))

f = next(x for x in snap.entity("orders").fields if x.name == "ord_sts")
print(
    f.name, f.type, "| distinct:", f.distinct_count, "|", [(v.value, v.count) for v in f.top_values]
)

7 tables, 42 fields, 8 relationships


,table,rows,columns
0,categories,15,3
1,customers,500,8
2,order_items,6252,6
3,orders,3000,6
4,payments,2958,6
5,products,120,7
6,reviews,1184,6


,relationship
0,categories.parent_category_id -> categories.ca...
1,order_items.order_id -> orders.order_id
2,order_items.product_id -> products.product_id
3,orders.customer_id -> customers.customer_id
4,payments.order_id -> orders.order_id
5,products.category_id -> categories.category_id
6,reviews.customer_id -> customers.customer_id
7,reviews.product_id -> products.product_id


ord_sts CHAR(1) | distinct: 5 | [('D', 2312), ('C', 372), ('R', 239), ('P', 42), ('S', 35)]


#### Describer Agent

In [12]:
DESCRIBER_SYSTEM = """You document database tables for a text-to-SQL system. You receive ONE table: its columns,
types, keys, row count, and for each column either the full list of distinct values with counts (low
cardinality) or a few example values. Write descriptions that help another model choose the right tables
and columns for a business question.

Rules:
- Be factual and concise: one sentence per column, at most 25 words.
- For code-like columns (short codes such as A, I, B) fill value_meanings using only the column name and the
values you were given. If you are not certain, give your best guess and set confidence to "low" or
"medium". Never use "high" for a guessed code meaning.
- Mark personal data (names, emails, phone numbers, addresses) with sensitive=true and
semantic_type="personal_data".
- Do not invent columns, business rules or facts that are not supported by the input.
- semantic_type must be one of: id, code, category, measure, timestamp, date, text, personal_data, flag, other.
- confidence must be one of: high, medium, low.

Return ONLY a JSON object of this shape:
{"table_description": "...",
"columns": [{"name": "...", "description": "...", "semantic_type": "...", "sensitive": false,
            "value_meanings": null, "confidence": "..."}]}
Include every column of the table exactly once. value_meanings is an object mapping code to meaning, or null."""

In [13]:
def describe_table(
    snapshot: SchemaSnapshot, table: str, *, use_cache: bool = True
) -> TableDescription:
    ent = snapshot.entity(table)
    payload = {
        "table": ent.name,
        "row_count": ent.row_count,
        "other_tables_in_database": [e.name for e in snapshot.entities if e.name != table],
        "columns": [
            {
                "name": f.name,
                "type": f.type,
                "nullable": f.nullable,
                "primary_key": f.primary_key,
                "references": f.references,
                "distinct_count": f.distinct_count,
                "values": [{"value": v.value, "count": v.count} for v in f.top_values],
                "personal_data_values_hidden": f.sensitive,
            }
            for f in ent.fields
        ],
    }
    messages = [
        {"role": "system", "content": DESCRIBER_SYSTEM},
        {"role": "user", "content": "Table to document (JSON):\n" + json.dumps(payload, indent=1)},
    ]
    expected = {f.name for f in ent.fields}
    last_error = None
    for _ in range(3):  # initial try plus up to 2 repairs
        raw = llm(messages, json_mode=True, use_cache=use_cache)
        try:
            desc = TableDescription.model_validate_json(raw)
            missing = expected - {c.name for c in desc.columns}
            if missing:
                raise ValueError(f"missing columns: {sorted(missing)}")
            return desc
        except Exception as e:
            last_error = e
            messages = messages + [
                {"role": "assistant", "content": raw},
                {
                    "role": "user",
                    "content": f"That output is invalid: {e}. Return corrected JSON only.",
                },
            ]
    raise RuntimeError(f"Describer failed for {table}: {last_error}")


In [14]:
def apply_description(snapshot: SchemaSnapshot, table: str, desc: TableDescription) -> None:
    ent = snapshot.entity(table)
    ent.description = desc.table_description
    by_name = {c.name: c for c in desc.columns}
    for f in ent.fields:
        c = by_name[f.name]
        f.description = c.description
        f.semantic_type = c.semantic_type
        f.sensitive = f.sensitive or c.sensitive
        f.value_meanings = c.value_meanings
        f.confidence = c.confidence


def describe_schema(snapshot: SchemaSnapshot) -> SchemaSnapshot:
    for ent in snapshot.entities:
        apply_description(snapshot, ent.name, describe_table(snapshot, ent.name))
    return snapshot

In [15]:
snap = describe_schema(snap)

for e in snap.entities:
    print(f"{e.name}: {e.description}")

categories: Category hierarchy for products, each row defines a category and its optional parent category.
customers: Contains 500 customers with identifiers, contact info, location, signup date, and status code.
order_items: Contains line items for each order, linking products with quantity, unit price, and discount percentage.
orders: Orders placed by customers, including order date, channel, status, and total monetary amount.
payments: Records of individual payments linked to orders, including method, amount, timestamp, and status.
products: Products catalog containing product identifiers, category links, names, SKUs, pricing, cost, and active status.
reviews: Contains product reviews with rating, optional text, and creation timestamps, linked to products and customers.


In [16]:
rows = []
for e in snap.entities:
    for f in e.fields:
        rows.append(
            {
                "table": e.name,
                "column": f.name,
                "description": f.description,
                "type": f.semantic_type,
                "confidence": f.confidence,
                "sensitive": f.sensitive,
                "value_meanings": f.value_meanings,
            }
        )
review_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 90)
print("Needs a human look (not high confidence, or a code column):")
display(review_df[(review_df.confidence != "high") | (review_df.type == "code")])

EXPECTED_CODES = {
    ("customers", "cust_stat_cd"): {"A": "activ", "I": "inactiv", "B": "block"},
    ("orders", "ord_sts"): {"P": "pend", "S": "ship", "D": "deliver", "C": "cancel", "R": "return"},
    ("payments", "pmt_sts"): {"S": "succe", "F": "fail", "R": "refund"},
}
checks = []
for (t, c), exp in EXPECTED_CODES.items():
    field = next(x for x in snap.entity(t).fields if x.name == c)
    got = {k.upper(): v.lower() for k, v in (field.value_meanings or {}).items()}
    for code, fragment in exp.items():
        meaning = got.get(code) or ""
        checks.append(
            {
                "column": f"{t}.{c}",
                "code": code,
                "expected": fragment,
                "got": meaning or None,
                "ok": fragment in meaning,
            }
        )
display(pd.DataFrame(checks))
print("code accuracy:", round(sum(x["ok"] for x in checks) / len(checks), 2))
print(
    "flagged sensitive:",
    [f"{e.name}.{f.name}" for e in snap.entities for f in e.fields if f.sensitive],
)

Needs a human look (not high confidence, or a code column):


,table,column,description,type,confidence,sensitive,value_meanings
10,customers,cust_stat_cd,"Customer status code: A=Active, I=Inactive, B=Blocked.",code,low,False,"{'A': 'Active', 'I': 'Inactive', 'B': 'Blocked'}"
20,orders,ord_sts,Single‑letter code representing the order's current status.,code,low,False,"{'D': 'Delivered or Done', 'C': 'Cancelled', 'R': 'Returned', 'P': 'Pending', 'S': 'Sh..."
28,payments,pmt_sts,Single‑character payment status code.,code,low,False,"{'S': 'Success', 'R': 'Refunded', 'F': 'Failed'}"
32,products,sku,Stock Keeping Unit code uniquely identifying the product.,code,high,False,None


,column,code,expected,got,ok
0,customers.cust_stat_cd,A,activ,active,True
1,customers.cust_stat_cd,I,inactiv,inactive,True
2,customers.cust_stat_cd,B,block,blocked,True
3,orders.ord_sts,P,pend,pending,True
4,orders.ord_sts,S,ship,shipped,True
5,orders.ord_sts,D,deliver,delivered or done,True
6,orders.ord_sts,C,cancel,cancelled,True
7,orders.ord_sts,R,return,returned,True
8,payments.pmt_sts,S,succe,success,True
9,payments.pmt_sts,F,fail,failed,True


code accuracy: 1.0
flagged sensitive: ['customers.full_name', 'customers.email', 'customers.phone']


In [17]:
REVIEW = {
    "customers.cust_stat_cd": {"value_meanings": {"A": "active", "I": "inactive", "B": "blocked"}},
    "orders.ord_sts": {
        "value_meanings": {
            "P": "pending",
            "S": "shipped",
            "D": "delivered",
            "C": "cancelled",
            "R": "returned",
        }
    },
    "payments.pmt_sts": {"value_meanings": {"S": "success", "F": "failed", "R": "refunded"}},
    # "orders.total_amount": {"description": "Order total in INR after item discounts."},
}

for key, changes in REVIEW.items():
    t, c = key.split(".")
    field = next(x for x in snap.entity(t).fields if x.name == c)
    for attr, value in changes.items():
        setattr(field, attr, value)
    field.confidence = "high"
    field.reviewed = True

for e in snap.entities:
    e.reviewed = True

SNAP_PATH = Path(settings.llm_cache_dir).parent / "demo_snapshot.json"
SNAP_PATH.write_text(snap.model_dump_json(indent=2), encoding="utf-8")
print("saved:", SNAP_PATH)

# to reload later without re-running the Describer:
# snap = SchemaSnapshot.model_validate_json(SNAP_PATH.read_text(encoding="utf-8"))

saved: D:\resume-projects\askdata\backend\.cache\demo_snapshot.json


### Indexing and vector baseline

In [18]:
from sqlalchemy import event


@event.listens_for(engine, "connect")
def _set_search_path(dbapi_conn, _record):
    cur = dbapi_conn.cursor()
    cur.execute("set search_path to askdata, extensions, public")  # so the vector type is found
    cur.close()
    dbapi_conn.commit()


engine.dispose()  # drop old pooled connections so new ones pick up the search_path

EMB_DIM = len(embed(["dimension probe"])[0])
print("embedding dimension:", EMB_DIM)

with engine.begin() as conn:
    conn.execute(text("create schema if not exists askdata"))
    conn.execute(
        text(f"""
        create table if not exists askdata.schema_chunks (
            id bigserial primary key,
            connection_id text not null,
            entity text not null,
            field text,
            kind text not null,
            content text not null,
            embedding vector({EMB_DIM}) not null,
            tsv tsvector generated always as (to_tsvector('simple', content)) stored,
            created_at timestamptz not null default now()
        )""")
    )
    conn.execute(
        text(
            "create index if not exists schema_chunks_embedding_idx "
            "on askdata.schema_chunks using hnsw (embedding vector_cosine_ops)"
        )
    )
    conn.execute(
        text(
            "create index if not exists schema_chunks_tsv_idx "
            "on askdata.schema_chunks using gin (tsv)"
        )
    )
    conn.execute(
        text(
            "create index if not exists schema_chunks_conn_idx "
            "on askdata.schema_chunks (connection_id)"
        )
    )

display(sql("select cast('[1,2,3]' as vector) <=> cast('[1,2,4]' as vector) as cosine_distance"))

embedding dimension: 384


,cosine_distance
0,0.00854


In [19]:
from collections import defaultdict


def build_chunks(snapshot: SchemaSnapshot) -> list[dict]:
    neighbors = defaultdict(set)
    for r in snapshot.relationships:
        neighbors[r.source].add(r.target)
        neighbors[r.target].add(r.source)

    chunks = []
    for e in snapshot.entities:
        chunks.append(
            {
                "entity": e.name,
                "field": None,
                "kind": "table",
                "content": (
                    f"Table {e.name} ({e.row_count} rows): {e.description} "
                    f"Columns: {', '.join(f.name for f in e.fields)}. "
                    f"Related tables: {', '.join(sorted(neighbors[e.name])) or 'none'}."
                ),
            }
        )
        for f in e.fields:
            c = f"{e.name}.{f.name} ({f.type}"
            if f.primary_key:
                c += ", primary key"
            if f.references:
                c += f", references {f.references}"
            c += f"): {f.description}"
            if f.value_meanings:
                c += " Values: " + ", ".join(f"{k}={v}" for k, v in f.value_meanings.items()) + "."
            elif f.top_values and not f.sensitive and (f.distinct_count or 99) <= 25:
                c += " Values: " + ", ".join(v.value for v in f.top_values) + "."
            chunks.append({"entity": e.name, "field": f.name, "kind": "column", "content": c})
    return chunks


chunks = build_chunks(snap)
print(len(chunks), "chunks | longest:", max(len(c["content"].split()) for c in chunks), "words")
for c in chunks[:3]:
    print("-", c["content"])
print("-", next(c["content"] for c in chunks if c["field"] == "ord_sts"))

49 chunks | longest: 44 words
- Table categories (15 rows): Category hierarchy for products, each row defines a category and its optional parent category. Columns: category_id, name, parent_category_id. Related tables: categories, products.
- categories.category_id (INTEGER, primary key): Unique identifier for each category. Values: 1, 9, 4, 8, 12, 10, 7, 13.
- categories.name (TEXT): Descriptive name of the category. Values: Fitness, Furniture, Electronics, Sports, Home & Kitchen, Audio, Books, Fashion.
- orders.ord_sts (CHAR(1)): Single‑letter code representing the order's current status. Values: P=pending, S=shipped, D=delivered, C=cancelled, R=returned.


In [20]:
def index_schema(snapshot: SchemaSnapshot) -> int:
    """Embed every chunk and replace the stored chunks for this connection."""
    chunks = build_chunks(snapshot)
    vecs = embed([c["content"] for c in chunks])
    rows = [
        {
            "cid": snapshot.connection_id,
            "entity": c["entity"],
            "field": c["field"],
            "kind": c["kind"],
            "content": c["content"],
            "emb": "[" + ",".join(str(x) for x in v) + "]",
        }
        for c, v in zip(chunks, vecs)
    ]
    with engine.begin() as conn:
        conn.execute(
            text("delete from askdata.schema_chunks where connection_id = :cid"),
            {"cid": snapshot.connection_id},
        )
        conn.execute(
            text("""
            insert into askdata.schema_chunks (connection_id, entity, field, kind, content, embedding)
            values (:cid, :entity, :field, :kind, :content, cast(:emb as vector))"""),
            rows,
        )
    return len(rows)


print("indexed chunks:", index_schema(snap))
display(sql("select kind, count(*) as chunks from askdata.schema_chunks group by kind"))

indexed chunks: 49


,kind,chunks
0,column,42
1,table,7


In [21]:
def search_vector(question: str, connection_id: str = "demo", k: int = 20) -> pd.DataFrame:
    qv = embed_query(question)
    q = "[" + ",".join(str(x) for x in qv) + "]"
    return sql(
        """
        select entity, field, kind, content,
               1 - (embedding <=> cast(:q as vector)) as score
        from askdata.schema_chunks
        where connection_id = :cid
        order by embedding <=> cast(:q as vector)
        limit :k""",
        q=q,
        cid=connection_id,
        k=k,
    )


hits = search_vector("Who are our top 5 customers by revenue?", k=8)
display(hits[["entity", "field", "kind", "score"]])

,entity,field,kind,score
0,customers,NaN,table,0.379724
1,customers,city,column,0.376165
2,customers,full_name,column,0.363414
3,customers,customer_id,column,0.363084
4,orders,NaN,table,0.328251
5,order_items,NaN,table,0.327922
6,customers,phone,column,0.316867
7,customers,state,column,0.316518


In [22]:
RETRIEVAL_QS = [
    {
        "q": "How many orders were delivered each month?",
        "tables": {"orders"},
        "columns": {"orders.ord_sts", "orders.order_date"},
    },
    {
        "q": "Who are our top 5 customers by revenue?",
        "tables": {"customers", "orders"},
        "columns": {"orders.total_amount", "customers.full_name"},
    },
    {
        "q": "Which products have the best average rating?",
        "tables": {"products", "reviews"},
        "columns": {"reviews.rating", "products.name"},
    },
    {
        "q": "What is the total revenue by product category?",
        "tables": {"order_items", "products", "categories"},
        "columns": {"order_items.qty", "order_items.unit_price", "categories.name"},
    },
    {
        "q": "What is the failed payment rate for each payment method?",
        "tables": {"payments"},
        "columns": {"payments.pmt_sts", "payments.method"},
    },
    {
        "q": "How many customers are blocked?",
        "tables": {"customers"},
        "columns": {"customers.cust_stat_cd"},
    },
    {
        "q": "What is the average discount given on each sales channel?",
        "tables": {"orders", "order_items"},
        "columns": {"order_items.disc_pct", "orders.channel"},
    },
    {
        "q": "Which city has the most active customers?",
        "tables": {"customers"},
        "columns": {"customers.city", "customers.cust_stat_cd"},
    },
    {
        "q": "Show orders that were returned and refunded.",
        "tables": {"orders", "payments"},
        "columns": {"orders.ord_sts", "payments.pmt_sts"},
    },
    {
        "q": "What is the profit margin for each product?",
        "tables": {"products"},
        "columns": {"products.unit_price", "products.cost_amt"},
    },
    {
        "q": "What percentage of orders were paid using UPI?",
        "tables": {"payments"},
        "columns": {"payments.method"},
    },
    {
        "q": "Which customers signed up in 2025 but never placed an order?",
        "tables": {"customers", "orders"},
        "columns": {"customers.signup_date", "orders.customer_id"},
    },
    {
        "q": "What is the average order value by state?",
        "tables": {"orders", "customers"},
        "columns": {"orders.total_amount", "customers.state"},
    },
    {
        "q": "Which products are no longer active?",
        "tables": {"products"},
        "columns": {"products.is_active"},
    },
    {
        "q": "List all 1-star reviews with the customer name.",
        "tables": {"reviews", "customers"},
        "columns": {"reviews.rating", "customers.full_name", "reviews.review_text"},
    },
]


def evaluate_retriever(retriever, label: str, k_tables: int = 4, k_cols: int = 10) -> pd.DataFrame:
    rows = []
    for item in RETRIEVAL_QS:
        out = retriever(item["q"])
        top_t = out["tables"][:k_tables]
        top_c = out["columns"][:k_cols]
        rows.append(
            {
                "question": item["q"],
                "table_recall": len(item["tables"] & set(top_t)) / len(item["tables"]),
                "all_tables": item["tables"] <= set(top_t),
                "col_recall": len(item["columns"] & set(top_c)) / len(item["columns"]),
                "missing_columns": sorted(item["columns"] - set(top_c)),
            }
        )
    df = pd.DataFrame(rows)
    print(
        f"{label}: table recall@{k_tables} = {df.table_recall.mean():.2f} | "
        f"all tables found = {df.all_tables.mean():.2f} | "
        f"column recall@{k_cols} = {df.col_recall.mean():.2f}"
    )
    return df


def vector_retriever(question: str, k: int = 20) -> dict:
    hits = search_vector(question, k=k)
    tables = list(dict.fromkeys(hits.entity))
    columns = [f"{e}.{f}" for e, f in zip(hits.entity, hits.field) if f]
    return {"tables": tables, "columns": columns}


baseline = evaluate_retriever(vector_retriever, "vector only")
pd.set_option("display.max_colwidth", 70)
display(baseline[(~baseline.all_tables) | (baseline.col_recall < 1)])

vector only: table recall@4 = 0.97 | all tables found = 0.93 | column recall@10 = 0.82


,question,table_recall,all_tables,col_recall,missing_columns
1,Who are our top 5 customers by revenue?,1.0,True,0.500000,[orders.total_amount]
3,What is the total revenue by product category?,1.0,True,0.333333,"[categories.name, order_items.qty]"
10,What percentage of orders were paid using UPI?,1.0,True,0.000000,[payments.method]
12,What is the average order value by state?,0.5,False,0.500000,[customers.state]
